In [1]:
#!/usr/bin/env python3
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.io.vasp.inputs import Poscar
from copy import deepcopy

In [2]:
# ===================== CONFIGURACIÓN =====================
poscar_sustrato = "Au_POSCAR"           # Bulk
poscar_2D = "POSCAR1"  # Slab

matrix_sustrato = np.array([[2, 0, 0],
                     [0, 2, 0],
                     [0, 0, 3]])

matrix_2D = np.array([[1, 0, 0],
                     [0, 1, 0],
                     [0, 0, 3]])
elements = ["Cs", "Pb", "Br"]
vacuum_between = 4.0   # entre bulk y slab (Å)
join_direction = 'z'



In [3]:
def mover_monocapa_pbc(s1, delta_xy):
    """
    Mueve toda la monocapa de manera rígida para que el átomo I quede sobre el target_point.
    Aplica condiciones periódicas de contorno para que todos los átomos queden dentro de la celda.
    """
    s1_shifted = deepcopy(s1)
    coords_cart = np.array(s1_shifted.cart_coords)

    # Aplicar desplazamiento solo en XY
    coords_cart[:, 0] += delta_xy[0]
    coords_cart[:, 1] += delta_xy[1]

    # Convertir a coordenadas fraccionarias para aplicar PBC
    coords_frac = s1_shifted.lattice.get_fractional_coords(coords_cart)

    # Aplicar módulo 1 para mantener los átomos dentro de la celda
    coords_frac = coords_frac % 1.0

    # Crear la nueva estructura con coordenadas cartesianas dentro de la celda
    s1_shifted = Structure(
        lattice=s1_shifted.lattice,
        species=s1_shifted.species,
        coords=s1_shifted.lattice.get_cartesian_coords(coords_frac),
        coords_are_cartesian=True
    )

    return s1_shifted

# Preparacion de sustrato y superficies antes de adaptar entre ellas

In [4]:
# ===================== LEER Y REPLICAR =====================
sustrato = Structure.from_file(poscar_sustrato)
s2D = Structure.from_file(poscar_2D)

sustrato.make_supercell(matrix_sustrato)
s2D.make_supercell(matrix_2D)

cellsustrato = sustrato.lattice.matrix
cell2D = s2D.lattice.matrix

# También podés obtener las longitudes y ángulos
print("\nLongitudes (Å):")
print("a1, b1, c1 =", sustrato.lattice.abc)
print("a2, b2, c2 =", s2D.lattice.abc)
print("Δa, Δb, Δc =", np.array(s2D.lattice.abc) - np.array(sustrato.lattice.abc))

print("\nÁngulos (°):")
print("α1, β1, γ1 =", sustrato.lattice.angles)
print("α2, β2, γ2 =", s2D.lattice.angles)
print("Δα, Δβ, Δγ =", np.array(s2D.lattice.angles) - np.array(sustrato.lattice.angles))

# # Guardar estructuras replicadas
# Poscar(sustrato).write_file("POSCAR_A_replicada.vasp")
# Poscar(s2D).write_file("POSCAR_B_replicada.vasp")


Longitudes (Å):
a1, b1, c1 = (8.342577062949454, 8.342577062949454, 12.513865594424182)
a2, b2, c2 = (8.10107841, 8.45384704, 35.63673369)
Δa, Δb, Δc = [-0.24149865  0.11126998 23.1228681 ]

Ángulos (°):
α1, β1, γ1 = (90.0, 90.0, 90.0)
α2, β2, γ2 = (90.0, 90.0, 90.0)
Δα, Δβ, Δγ = [0. 0. 0.]


In [5]:
perovskite_para_cut=s2D
# interface_centered es tu estructura
coords = np.array(perovskite_para_cut.cart_coords)
species = np.array([str(s) for s in perovskite_para_cut.species])
# Especies de interés

# Encontrar z máxima para cada especie
z_max = {}
for el in elements:
    z_max[el] = coords[species == el, 2].max()
    
print("Alturas máximas por especie (z):", z_max)

# Encontrar z máxima para cada especie
z_min = {}
for el in elements:
    z_min[el] = coords[species == el, 2].min()
    
print("Alturas máximas por especie (z):", z_min)

Alturas máximas por especie (z): {'Cs': 32.6670058825, 'Pb': 29.697278075, 'Br': 35.260156999002625}
Alturas máximas por especie (z): {'Cs': 2.969727807500001, 'Pb': 2.5e-16, 'Br': 0.376576690997363}


In [6]:
# Cortes definidos (z máximo permitido para cada especie)
cutoff_z_max = {"Cs": 30.20, "Pb": 30.20, "Br": 30.20}
cutoff_z_min = {"Cs": 3.2, "Pb": 3.2, "Br": 3.2}

# Crear máscara para seleccionar átomos
mask = np.ones(len(coords), dtype=bool)  # por defecto todos True

for el, z_max in cutoff_z_max.items():
    idx = (species == el) & (coords[:, 2] > z_max)  # átomos a eliminar
    mask[idx] = False
for el, z_min in cutoff_z_min.items():
    idx = (species == el) & (coords[:, 2] < z_min)  # átomos a eliminar
    mask[idx] = False



# Aplicar máscara
new_coords = coords[mask]
new_species = [species[i] for i in range(len(species)) if mask[i]]

# Crear nueva estructura
s2D = Structure(
    lattice=perovskite_para_cut.lattice,
    species=new_species,
    coords=new_coords,
    coords_are_cartesian=True
)

In [7]:
# Obtener coordenadas cartesianas originales de s1 y s2
coords_sustrato = np.array([site.coords for site in sustrato])
coords_2D = np.array([site.coords for site in s2D])
# Rotación de la supercelda de s1 para que a_vec quede en X y b_vec perpendicular
a_vec_sustrato = sustrato.lattice.matrix[0]
b_vec_sustrato = sustrato.lattice.matrix[1]
c_vec_sustrato = sustrato.lattice.matrix[2]

# Ángulo que hace a_vec con X
theta = -np.arctan2(a_vec_sustrato[1], a_vec_sustrato[0])
cos_t = np.cos(theta)
sin_t = np.sin(theta)

# Matriz de rotación en XY
R_xy = np.array([[cos_t, -sin_t, 0],
                 [sin_t,  cos_t, 0],
                 [0,     0,      1]])

# Rotar vectores de la celda
a_vec_rot = R_xy @ a_vec_sustrato
b_vec_rot = R_xy @ b_vec_sustrato
c_vec_rot = c_vec_sustrato  # no cambiamos Z

# Nueva celda alineada
new_lattice = Lattice([a_vec_rot, b_vec_rot, c_vec_rot])
coords_sustrato_rot = (R_xy @ coords_sustrato.T).T
# Crear un nuevo objeto Structure con la misma especie y coordenadas
sustrato_aligned = Structure(
    lattice=new_lattice,
    species=sustrato.species,
    coords=coords_sustrato_rot,
    coords_are_cartesian=True
)

In [8]:
# ===================== COORDENADAS PARA INTERFAZ =====================
coords_2D_base_sustrato = coords_2D.copy()
# Calcular extremos en z
max_z1 = np.max(coords_sustrato_rot[:, 2])
min_z2 = np.min(coords_2D_base_sustrato[:, 2])

# Desplazar slab encima del bulk + vacuum_between
shift_z = max_z1 - min_z2 + vacuum_between
coords_2D_base_sustrato[:, 2] += shift_z

# ===================== COMBINAR POSICIONES Y ESPECIES =====================
combined_species = list(sustrato_aligned.species) + list(s2D.species)
combined_coords = np.vstack([coords_sustrato_rot, coords_2D_base_sustrato])

# ===================== DEFINIR NUEVA CELDA =====================
a_vec, b_vec, c_vec_bulk = sustrato_aligned.lattice.matrix

height1 = np.max(coords_sustrato_rot[:,2]) - np.min(coords_sustrato_rot[:,2])
height2 = np.max(coords_2D_base_sustrato[:,2]) - np.min(coords_2D_base_sustrato[:,2])
new_c_length = height1 + height2 + vacuum_between + vacuum_between
c_vec_unit = c_vec_bulk / np.linalg.norm(c_vec_bulk)
new_c_vec = c_vec_unit * new_c_length

new_lattice = Lattice([a_vec, b_vec, new_c_vec])

s_2D_base_sustrato = Structure(
    lattice=new_lattice,              # o podés usar Lattice(cell2) si querés mantener su celda original
    species=s2D.species,
    coords=coords_2D_base_sustrato,
    coords_are_cartesian=True
)
interface = Structure(
    lattice=new_lattice,
    species=combined_species,
    coords=combined_coords,
    coords_are_cartesian=True
)
Poscar(interface).write_file("POSCAR.vasp")
print("✅ Interfaz centrada guardada en POSCAR_interface_centered.vasp")

✅ Interfaz centrada guardada en POSCAR_interface_centered.vasp


## Info para hacer desplazamientos

In [9]:
au_sites = [site for site in interface if site.specie.symbol == "Au"]
coords_au = np.array([site.coords for site in au_sites])
# 1️⃣ Encontrar el átomo de Au más alto (mayor z)
idx_top = np.argmax(coords_au[:, 2])
top_atom = au_sites[idx_top]
coord_top = coords_au[idx_top]
print(f"Átomo de Au más alto: índice {idx_top}, z = {coord_top[2]:.3f}")

Átomo de Au más alto: índice 17, z = 10.428


In [10]:
# 2️⃣ Definir tolerancia en z (para excluir la camada de abajo)
z_tol = 0.5  # puedes ajustar este valor según tu estructura
# Calcular diferencias
delta = coords_au - coord_top
# Filtrar solo los átomos que están en la misma capa (diferencia en z menor a z_tol)
mask_same_layer = np.abs(delta[:, 2]) < z_tol
# Excluir el mismo átomo
mask_same_layer[idx_top] = False
# Si no hay vecinos en esa capa, avisar
if not np.any(mask_same_layer):
    raise ValueError("No se encontraron vecinos en la misma capa. Aumenta z_tol.")

# Calcular distancias XY solo para los átomos de la misma capa
delta_xy = delta[mask_same_layer].copy()
delta_xy[:, 2] = 0.0
dist_xy = np.linalg.norm(delta_xy, axis=1)
# Índice del vecino más cercano en la lista filtrada
idx_neighbor_in_mask = np.argmin(dist_xy)

# Convertir a índice original
idx_neighbor = np.where(mask_same_layer)[0][idx_neighbor_in_mask]

neighbor_atom = au_sites[idx_neighbor]
coord_neighbor = coords_au[idx_neighbor]
print(f"Vecino más cercano en la misma capa: índice {idx_neighbor}, z = {coord_neighbor[2]:.3f}, distancia_xy = {dist_xy[idx_neighbor_in_mask]:.3f}")


Vecino más cercano en la misma capa: índice 29, z = 10.428, distancia_xy = 2.950


In [11]:
# Filtrar solo los átomos de Br
br_sites = [site for site in interface if site.specie.symbol == "Pb"]
# Obtener coordenadas cartesianas
coords_br = np.array([site.coords for site in br_sites])
# 1️⃣ Encontrar el átomo de Br más bajo (menor z)
idx_bottom = np.argmin(coords_br[:, 2])
br_bottom_atom = br_sites[idx_bottom]
coord_br_bottom = coords_br[idx_bottom]
print(f"Átomo de Br más bajo: índice {idx_bottom}, z = {coord_br_bottom[2]:.3f}")


Átomo de Br más bajo: índice 0, z = 14.805


In [12]:

# ==== Coordenadas cartesianas ====
I_coord = interface[idx_bottom].coords
Au1_coord = interface[idx_top].coords
Au2_coord = interface[idx_neighbor].coords

# ==== Puntos objetivo ====
target_A = Au1_coord                    # encima de Au1
target_B = 0.5 * (Au1_coord + Au2_coord)  # punto medio entre Au1 y Au2
delta_A =  target_A[:2] - I_coord[:2]
delta_B =  target_B[:2] - I_coord[:2]
print(delta_A,delta_B)


[1.0500000e-15 6.2569328e+00] [1.04282213 5.21411066]


## top site

In [13]:
print(delta_A)
monocapa_sobre_Au1 = mover_monocapa_pbc(s_2D_base_sustrato,delta_A)
sites_m = [site for site in monocapa_sobre_Au1]
coords_monocamada = np.array([site.coords for site in sites_m])

# ===================== COMBINAR POSICIONES Y ESPECIES =====================
combined_species = list(sustrato_aligned.species) + list(s2D.species)
combined_coords = np.vstack([coords_sustrato_rot, coords_monocamada])
s_2D_base_sustrato_top = Structure(
    lattice=new_lattice,              # o podés usar Lattice(cell2) si querés mantener su celda original
    species=s2D.species,
    coords=coords_monocamada,
    coords_are_cartesian=True
)
interface_top = Structure(
    lattice=new_lattice,
    species=combined_species,
    coords=combined_coords,
    coords_are_cartesian=True
)
# Guardar POSCAR final
Poscar(interface_top).write_file("POSCAR_top.vasp")

[1.0500000e-15 6.2569328e+00]


## bridge site

In [14]:
print(delta_B)
monocapa_entre_Au1_Au2 = mover_monocapa_pbc(s_2D_base_sustrato,delta_B)
sites_m = [site for site in monocapa_entre_Au1_Au2]
coords_monocamada = np.array([site.coords for site in sites_m])

# ===================== COMBINAR POSICIONES Y ESPECIES =====================
combined_species = list(sustrato_aligned.species) + list(s2D.species)
combined_coords = np.vstack([coords_sustrato_rot, coords_monocamada])
s_2D_base_sustrato_bridge = Structure(
    lattice=new_lattice,              # o podés usar Lattice(cell2) si querés mantener su celda original
    species=s2D.species,
    coords=coords_monocamada,
    coords_are_cartesian=True
)
interface_bridge = Structure(
    lattice=new_lattice,
    species=combined_species,
    coords=combined_coords,
    coords_are_cartesian=True
)

# Guardar POSCAR final
Poscar(interface_bridge).write_file("POSCAR_bridge.vasp")

[1.04282213 5.21411066]
